# Building a Reusable Actuarial Pricing Library with PROC FCMP

## Executive Summary

A property-and-casualty insurer encapsulates its core pricing math — pure premium, expense/risk loading, limited-fluctuation credibility blending, and discounted reserve estimation — as custom functions and a multi-output subroutine in **PROC FCMP**. The compiled library is registered through the `CMPLIB=` system option and then called row-by-row from a DATA step that rates a synthetic 2,000-policy portfolio. Every rated figure in this notebook — the credibility factor `z`, the blended pure premium, the charged premium, and the present-valued case reserve — is produced by these compiled routines, not by inline arithmetic. The result: the implied loss ratio lands at 58.8% (rural), 66.6% (suburban), and 70.6% (urban) — comfortably below 100% in every segment, confirming the loaded premium covers expected loss while the rating step stays clean and auditable.

## Data Sources

| Dataset | Rows | Description | Key Variables |
|---------|------|-------------|---------------|
| `work.portfolio` | 2,000 | Synthetic in-force P&C policy portfolio generated inline with `rand()` | `policy_id`, `region` (urban/suburban/rural), `years_insured`, `exposure` (car-years), `claim_count` (Poisson), `incurred_loss` (gamma severity x count), `class_pure_prem` (manual rate by region) |

The DATA step loops `policy_id` from 10001 to 12000, generating the full **2,000-policy** book that is rated below. The region mix falls out of the simulated assignment thresholds: 515 rural, 667 suburban, and 818 urban policies.

# Building a Reusable Actuarial Pricing Library with PROC FCMP

Actuaries repeat the same calculations across pricing, reserving, and reporting jobs: convert losses to a *pure premium*, apply *expense and risk loadings* to reach a charged premium, blend an individual policy's own experience with a class rate using *credibility*, and *discount* future cash flows to present value. Re-typing these formulas in every DATA step invites copy-paste errors and makes assumption changes painful.

**PROC FCMP** (the SAS function compiler) lets us define each formula once as a named function or subroutine, store the compiled routines in a library, and then call them like any built-in SAS function. In this notebook we:

1. Compile a small actuarial function library with `PROC FCMP`.
2. Register it for the session with the `CMPLIB=` system option.
3. Generate a synthetic property-and-casualty portfolio.
4. Rate every policy by calling our custom functions and a multi-output subroutine from a single DATA step.
5. Summarize and interpret the rated portfolio.

## Step 1 — Generate a synthetic policy portfolio

We simulate a book of 2,000 in-force auto policies. Each policy belongs to a rating region with its own manual *pure premium* (expected loss per car-year). Claim counts follow a Poisson process scaled by exposure, and severities follow a gamma distribution, so the `incurred_loss` is a realistic compound (Poisson x gamma) loss. `years_insured` will later drive the credibility weight. A fixed seed via `call streaminit` makes the data reproducible.

In [1]:
data portfolio;
    call streaminit(20260531);
    length region $9;
    do policy_id = 10001 to 12000;
        /* Assign a rating region */
        u = rand('uniform');
        if u < 0.40 then do; region = 'urban';    base_pp = 820; lambda = 0.18; end;
        else if u < 0.75 then do; region = 'suburban'; base_pp = 540; lambda = 0.11; end;
        else do; region = 'rural';    base_pp = 360; lambda = 0.07; end;

        /* Tenure (years insured) and exposure (car-years) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Compound claim process: Poisson frequency, gamma severity */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        do c = 1 to claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        end;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manual class pure premium for this policy's exposure */
        class_pure_prem = round(base_pp * exposure, 0.01);

        output;
    end;
    keep policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
run;

proc print data=portfolio(obs=8) noobs;
    title 'First 8 Simulated Policies';
run;

                                               First 8 Simulated Policies                                               

POLICY_ID    REGION  YEARS_INSURED  EXPOSURE  CLAIM_COUNT  INCURRED_LOSS  CLASS_PURE_PREM
    10001  rural                 5      1.36            0              0            489.6
    10002  urban                 8      0.97            1        3432.28            795.4
    10003  urban                 2      1.53            2        7155.92           1254.6
    10004  suburban              9       2.4            0              0             1296
    10005  rural                 5      0.99            0              0            356.4
    10006  suburban              1      0.83            0              0            448.2
    10007  rural                 5      0.97            0              0            349.2
    10008  rural                 7      2.32            0              0            835.2

... 1992 more observations (showing 8 of 2000)



NOTE: DATA portfolio


NOTE: Wrote portfolio (2000 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Step 2 — Compile the actuarial function library

Now the heart of the notebook. `PROC FCMP` with `OUTLIB=work.actfuncs.pricing` compiles four functions and one subroutine into the `pricing` package of the `work.actfuncs` data set:

- **`pure_premium`** — observed loss per unit of exposure (frequency x severity combined).
- **`credibility_z`** — limited-fluctuation credibility factor `Z = sqrt(n / (n + k))`, where `n` is the policy's exposure-years and `k` is a tuning constant.
- **`charged_premium`** — applies a proportional risk load and a fixed expense ratio to a loss cost to reach the premium we actually charge.
- **`pv_reserve`** — present value of a future claim payment, `FV / (1+r)**t`, used to discount case reserves.
- **`blend_premium`** (a SUBROUTINE) — uses `OUTARGS` to return *two* values at once: the credibility-weighted pure premium and the credibility factor it used, so the DATA step gets both in a single call.

Each routine is closed with `ENDSUB`, and the subroutine names its writable arguments with `OUTARGS`.

In [2]:
proc fcmp outlib=work.actfuncs.pricing;

    /* Observed pure premium: loss cost per unit of exposure */
    function pure_premium(loss, exposure);
        if exposure <= 0 then return(.);
        return(loss / exposure);
    endsub;

    /* Limited-fluctuation credibility: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        if n_years <= 0 then return(0);
        return(sqrt(n_years / (n_years + k)));
    endsub;

    /* Charged premium = loss cost grossed up for risk load + expenses */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        if expense_ratio >= 1 then return(.);
        return(gross / (1 - expense_ratio));
    endsub;

    /* Present value of a future claim payment discounted t years at rate r */
    function pv_reserve(future_value, r, t);
        return(future_value / (1 + r) ** t);
    endsub;

    /* Credibility blend: returns the weighted pure premium AND the Z used */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

run;


               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium



NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Step 3 — Register the library with CMPLIB=

Compiling the routines is not enough; SAS must be told where to find them when a later DATA step or procedure references a name it does not recognize as built-in. The `CMPLIB=` system option points at the data set (not the three-level package name) holding the compiled code. After this `OPTIONS` statement, every function and subroutine above is callable by name.

In [3]:
options cmplib=work.actfuncs;

NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Step 4 — Rate the portfolio with the custom functions

The rating DATA step is now almost free of arithmetic — the actuarial intent reads directly off the function names. For each policy we:

1. Compute the policy's own observed pure premium with `pure_premium`.
2. Call the `blend_premium` subroutine to credibility-weight it against the regional class rate, recovering both the blended loss cost and the credibility factor `z` through `OUTARGS`.
3. Gross the blended loss cost up to a charged premium with a 12% risk load and a 25% expense ratio via `charged_premium`.
4. Estimate the still-open case reserve as 35% of the policy's incurred loss and discount it three years at 4% to present value with `pv_reserve`.

Note how the subroutine's output arguments (`blended_pp`, `z`) must be initialized before the `CALL`. The reserve PV varies policy by policy because it is driven by each policy's actual incurred loss — claim-free policies carry a zero reserve, so their `reserve_pv` is zero too.

In [4]:
data rated;
    set portfolio;

    /* 1. Policy's own loss experience as a pure premium */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Credibility-weight own experience against the class rate.
          k = 4 exposure-years for full-ish credibility. */
    blended_pp = .;
    z = .;
    call blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Convert blended loss cost (per car-year) to charged premium */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Outstanding case reserve = 35% of incurred loss, expected to
          settle in 3 years; discount it to present value at 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    keep policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
run;

proc print data=rated(obs=10) noobs;
    title 'Rated Portfolio (first 10 policies)';
    var policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
run;

                                          Rated Portfolio (first 10 policies)                                           

POLICY_ID    REGION  YEARS_INSURED  EXPOSURE   OWN_PP  BLENDED_PP      Z  PREMIUM  RESERVE_PV
    10001  rural                 5      1.36        0       91.67  0.745   186.18           0
    10002  urban                 8      0.97  3538.43     3039.59  0.816  4402.95     1067.95
    10003  urban                 2      1.53  4677.07     3046.88  0.577  6961.51     2226.55
    10004  suburban              9       2.4        0       90.69  0.832   325.03           0
    10005  rural                 5      0.99        0       91.67  0.745   135.52           0
    10006  suburban              1      0.83        0       298.5  0.447   369.98           0
    10007  rural                 5      0.97        0       91.67  0.745   132.79           0
    10008  rural                 7      2.32        0       72.82  0.798   252.29           0
    10009  urban                

NOTE: DATA rated


NOTE: Read 2000 rows from portfolio.
NOTE: Wrote rated (2000 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.34 seconds
  cpu   0.34 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Step 5 — Summarize the rated book

With every policy priced through the same compiled library, we can roll the portfolio up by region. We report mean charged premium, mean credibility factor, total incurred loss, and the total present-valued case reserve, then derive the implied loss ratio per segment so we can see whether the loaded premium covers expected loss.

In [5]:
proc means data=rated mean sum maxdec=2 nonobs;
    class region;
    var premium z incurred_loss reserve_pv;
    title 'Portfolio Summary by Rating Region';
run;

/* Implied loss ratio = incurred loss / charged premium, plus the
   present-valued reserve the segment is carrying, by region. */
proc sql;
    title 'Implied Loss Ratio and Reserve PV by Region';
    select region,
           count(*)                              as n_policies,
           sum(incurred_loss)                    as total_loss     format=dollar12.2,
           sum(premium)                          as total_premium  format=dollar12.2,
           sum(incurred_loss) / sum(premium)     as loss_ratio     format=percent8.1,
           sum(reserve_pv)                       as total_pv_reserve format=dollar12.2
    from rated
    group by region
    order by region;
quit;
title;

                                           Portfolio Summary by Rating Region                                           

                                                  The MEANS Procedure

                                              Analysis Variable : premium

        region             Mean            Sum
        --------------------------------------
        rural            639.60      329393.72
        suburban        1202.44      802024.20
        urban           1975.95     1616324.50
        --------------------------------------

                                                 Analysis Variable : z

        region             Mean            Sum
        --------------------------------------
        rural              0.69         353.80
        suburban           0.68         456.89
        urban              0.68         558.33
        --------------------------------------

                                           Analysis Variable : incurred_loss

        region  

NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interpreting the results

The rating DATA step never spells out a single discount or credibility formula inline — it just calls `pure_premium`, `blend_premium`, `charged_premium`, and `pv_reserve`. That is the payoff of **PROC FCMP**: the actuarial assumptions live in one compiled library that can be unit-tested, version-controlled, and reused across pricing, reserving, and reporting jobs. Changing the credibility constant `k`, the risk load, or the expense ratio is a single edit in the library, not a hunt through every program.

Reading the rated book and the regional roll-up:

- **Credibility (`z`)** rises with `years_insured`, exactly as the limited-fluctuation formula `Z = sqrt(n / (n + k))` dictates. Among the first ten policies, the one-year suburban policy (10006) earns `z = 0.447`, the two-year urban policy (10003) `z = 0.577`, while the nine-year suburban policy (10004) reaches `z = 0.832`. Thin-experience policies are pulled toward the regional class rate; long-tenured ones lean on their own losses.
- **Blending in action:** most of the book is claim-free — only 332 of the 2,000 policies (16.6%) incurred a loss, so 1,668 carry `own_pp = 0` and `blend_premium` returns a `blended_pp` close to `(1 - z)` times the class rate. Among the first ten policies, 10001 (rural, `z = 0.745`) lands at `91.67` against a `360`/car-year class rate. The two displayed policies that did incur losses, 10002 and 10003, instead pull their blended loss cost up toward their own high experience (`3039.59` and `3046.88`).
- **Charged premium** sits above the blended loss cost because `charged_premium` adds a 12% risk load and grosses up for a 25% expense ratio, a fixed `1.12 / 0.75 = 1.493` multiplier on the loss cost. Mean charged premium runs `639.60` (rural), `1,202.44` (suburban), and `1,975.95` (urban) — the regional ordering tracks each segment's manual class rate and claim frequency.
- **Discounted reserves:** `pv_reserve` discounts each policy's outstanding case reserve (35% of incurred loss) three years at 4%, a `0.889` present-value factor. Claim-free policies carry `reserve_pv = 0`; among the displayed rows, 10002 and 10003 contribute `1,067.95` and `2,226.55`. Rolled up across all claimants, the book holds a present-valued reserve of `$60,234.95` (rural), `$166,111.86` (suburban), and `$354,854.95` (urban).
- **Implied loss ratios** land at 58.8% (rural), 66.6% (suburban), and 70.6% (urban) — all comfortably below 100%, so the loaded premium covers expected loss in every segment. The urban segment runs the hottest: it has the highest claim frequency (190 of 818 policies, 23.2%, incurred a loss versus 8.2% rural and 15.0% suburban), so its loss ratio sits above the others even though the same loading applies everywhere. A real rate review would test whether that frequency gap persists across more accident years before adjusting the manual rate.

The `blend_premium` subroutine also demonstrates the `OUTARGS` idiom for returning multiple results from one `CALL` — the blended loss cost and the credibility factor `z` come back together — avoiding separate function calls and keeping the per-policy rating logic compact and auditable.